## 04_scaling_analysis
  Log-log OLS scaling exponents for all Y variables against population, with analytic and
  bootstrap (n = 1000) confidence intervals; capital-city sensitivity.
  Input: nuts3_master_table.parquet. Output: results/model_summaries/*.csv, scatter plots.

In [ ]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

sys.path.append(str(Path.cwd().parent / "src"))
from scaling_stats import (
    fit_scaling_law,
    bootstrap_beta_ci,
    compare_ci_methods,
    plot_ci_comparison,
    plot_scaling_scatter,
)

DATA_DIR = Path.cwd().parent / "data" / "processed"
FIGURES_DIR = Path.cwd().parent / "figures"
RESULTS_DIR = Path.cwd().parent / "results"
plots_dir = Path.cwd().parent / "plots"
plots_dir.mkdir(exist_ok=True, parents=True)

df = pd.read_parquet(DATA_DIR / "nuts3_master_table.parquet")
df.shape
# Innovation-related POI columns replacing the all-objects poi_count
POI_COLS = ["poi_count_university", "poi_count_office", "poi_count_library"]

In [ ]:
print("Oszlopok és típusaik:")
print(df.dtypes)
print()

year_like = [c for c in df.columns if "year" in c.lower()]
pop_like = [c for c in df.columns if "pop" in c.lower()]

print(f"Év-jellegű oszlopok: {year_like}")
print(f"Populáció-jellegű oszlopok: {pop_like}")
print()

if year_like:
    print(f"'{year_like[0]}' egyedi értékei:")
    print(sorted(df[year_like[0]].dropna().unique()))

In [ ]:
gdp_comparison = compare_ci_methods(df, y_col="gdp_total")
gdp_comparison

In [ ]:
capital_nuts3 = {
    "HU110",  # Budapest
    "CZ010",  # Praha
    "SK010",  # Bratislavský kraj
    "PL911",  # Warszawa (m.st.)
}

print("Ellenőrzés — a fenti kódokra:")
print(df[df["nuts3_id"].isin(capital_nuts3)][["nuts3_id", "country_code"]].drop_duplicates())

In [ ]:
def compare_with_without_capitals(df, y_col, capital_ids, year_col="year", n_col="population", years=None):
    df_full = df
    df_no_capitals = df[~df["nuts3_id"].isin(capital_ids)]

    full_result = compare_ci_methods(df_full, y_col=y_col, year_col=year_col, n_col=n_col, years=years)
    no_cap_result = compare_ci_methods(df_no_capitals, y_col=y_col, year_col=year_col, n_col=n_col, years=years)

    merged = full_result.merge(
        no_cap_result,
        on="year", suffixes=("_full", "_no_capitals"),
    )
    return merged

In [ ]:
gdp_capital_check = compare_with_without_capitals(df, y_col="gdp_total", capital_ids=capital_nuts3)
gdp_capital_check

In [ ]:
capital_sensitivity_results = {}

capital_sensitivity_results["gdp_total"] = compare_with_without_capitals(
    df, y_col="gdp_total", capital_ids=capital_nuts3
)
capital_sensitivity_results["employment"] = compare_with_without_capitals(
    df, y_col="employment", capital_ids=capital_nuts3
)
capital_sensitivity_results["road_freight_total"] = compare_with_without_capitals(
    df, y_col="road_freight_total", capital_ids=capital_nuts3
)
capital_sensitivity_results["active_enterprises"] = compare_with_without_capitals(
    df, y_col="active_enterprises", capital_ids=capital_nuts3, years=[2021, 2022, 2023]
)

In [ ]:
for y_col, result in capital_sensitivity_results.items():
    print(f"\n=== {y_col} ===")
    display(result)

In [ ]:
df[["nuts3_id", "area"]].drop_duplicates().describe()

In [ ]:
area_fit_2023 = pd.DataFrame([fit_scaling_law(df[df["year"] == 2023], y_col="area", n_col="population")])
area_fit_2023


In [ ]:
capital_sensitivity_results["area"] = compare_with_without_capitals(
    df, y_col="area", capital_ids=capital_nuts3, years=[2023]
)
capital_sensitivity_results["area"]

In [ ]:
output_dir = RESULTS_DIR / "model_summaries"
output_dir.mkdir(exist_ok=True, parents=True)
output_path = output_dir / "capital_sensitivity_results.csv"

combined = pd.concat(
    [result.assign(y_variable=y_col) for y_col, result in capital_sensitivity_results.items()],
    ignore_index=True,
)
combined.to_csv(output_path, index=False)

print(f"Mentve: {output_path.relative_to(Path.cwd().parent)}")

In [ ]:
fig, ax = plot_scaling_scatter(df, y_col="gdp_total", n_col="population", year=2023)
plt.show()

In [ ]:
df_no_capitals = df[~df["nuts3_id"].isin(capital_nuts3)]
fig, ax = plot_scaling_scatter(df_no_capitals, y_col="gdp_total", n_col="population", year=2023)
plt.show()

In [ ]:
def plot_scaling_grid(
    df: pd.DataFrame,
    y_cols: list[str],
    capital_ids: set[str],
    n_col: str = "population",
    year: int | None = None,
    year_col: str = "year",
    id_col: str = "nuts3_id",
    figsize_per_panel: tuple[float, float] = (4.5, 4),
):
    """
    Bettencourt-stílusú log-log szórásdiagram-rács, több Y-változóra egyszerre.

    Két sor:
      - felső sor: teljes minta (fővárosok pirossal kiemelve)
      - alsó sor:  fővárosok nélküli minta, újraillesztett béta/R²-vel

    Oszlopok: y_cols listája szerint, egy oszlop / Y-változó.
    """
    data = df if year is None else df[df[year_col] == year]
    n_cols = len(y_cols)

    fig, axes = plt.subplots(
        2, n_cols,
        figsize=(figsize_per_panel[0] * n_cols, figsize_per_panel[1] * 2),
        squeeze=False,
    )

    for col_idx, y_col in enumerate(y_cols):
        # --- felső sor: teljes minta, fővárosok kiemelve ---
        ax_full = axes[0, col_idx]
        fit_full = fit_scaling_law(data, y_col=y_col, n_col=n_col)

        valid_full = data[[id_col, n_col, y_col]].dropna()
        valid_full = valid_full[(valid_full[n_col] > 0) & (valid_full[y_col] > 0)]
        is_capital = valid_full[id_col].isin(capital_ids)

        log_n_full = np.log(valid_full[n_col])
        log_y_full = np.log(valid_full[y_col])

        ax_full.scatter(
            log_n_full[~is_capital], log_y_full[~is_capital],
            color="tab:blue", edgecolor="white", s=40, zorder=3, label="régió",
        )
        ax_full.scatter(
            log_n_full[is_capital], log_y_full[is_capital],
            color="tab:red", edgecolor="white", s=60, zorder=4, label="főváros",
        )

        x_line = np.linspace(log_n_full.min(), log_n_full.max(), 100)
        ax_full.plot(x_line, fit_full["alpha"] + fit_full["beta"] * x_line,
                     color="black", linewidth=2, zorder=2)

        ax_full.text(
            0.05, 0.92, f"β={fit_full['beta']:.3f}   R²={fit_full['r_squared']:.2f}",
            transform=ax_full.transAxes, fontsize=11, va="top",
        )
        ax_full.set_title(f"{y_col} (teljes minta)")
        ax_full.set_xlabel(f"LN[{n_col}]")
        ax_full.set_ylabel(f"LN[{y_col}]")
        if col_idx == 0:
            ax_full.legend(loc="lower right", fontsize=9)

        # --- alsó sor: fővárosok nélkül ---
        ax_no_cap = axes[1, col_idx]
        data_no_cap = data[~data[id_col].isin(capital_ids)]
        fit_no_cap = fit_scaling_law(data_no_cap, y_col=y_col, n_col=n_col)

        valid_no_cap = data_no_cap[[n_col, y_col]].dropna()
        valid_no_cap = valid_no_cap[(valid_no_cap[n_col] > 0) & (valid_no_cap[y_col] > 0)]
        log_n_nc = np.log(valid_no_cap[n_col])
        log_y_nc = np.log(valid_no_cap[y_col])

        ax_no_cap.scatter(log_n_nc, log_y_nc, color="tab:blue", edgecolor="white", s=40, zorder=3)

        x_line_nc = np.linspace(log_n_nc.min(), log_n_nc.max(), 100)
        ax_no_cap.plot(x_line_nc, fit_no_cap["alpha"] + fit_no_cap["beta"] * x_line_nc,
                       color="black", linewidth=2, zorder=2)

        ax_no_cap.text(
            0.05, 0.92, f"β={fit_no_cap['beta']:.3f}   R²={fit_no_cap['r_squared']:.2f}",
            transform=ax_no_cap.transAxes, fontsize=11, va="top",
        )
        ax_no_cap.set_title(f"{y_col} (fővárosok nélkül)")
        ax_no_cap.set_xlabel(f"LN[{n_col}]")
        ax_no_cap.set_ylabel(f"LN[{y_col}]")

    fig.tight_layout()
    return fig, axes

In [ ]:
fig, axes = plot_scaling_grid(
    df[df["year"] == 2023],
    y_cols=["gdp_total", "employment", "road_freight_total"],
    capital_ids=capital_nuts3,
    n_col="population",
)
fig.savefig(plots_dir / "scaling_grid_gdp_employment_freight_2023.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
y_cols_to_plot = ["gdp_total", "employment", "road_freight_total", "active_enterprises"]

df_no_capitals = df[~df["nuts3_id"].isin(capital_nuts3)]

for y_col in y_cols_to_plot:
    fig, ax = plot_scaling_scatter(df, y_col=y_col, n_col="population", year=2023)
    ax.set_title(f"{y_col} (teljes minta, 2023)")
    plt.show()

    fig, ax = plot_scaling_scatter(df_no_capitals, y_col=y_col, n_col="population", year=2023)
    ax.set_title(f"{y_col} (fővárosok nélkül, 2023)")
    plt.show()

In [ ]:
years_by_variable = {
    "gdp_total": range(2014, 2024),
    "employment": range(2014, 2024),
    "road_freight_total": range(2014, 2024),
    "active_enterprises": range(2021, 2024),
}

output_dir = FIGURES_DIR / "scaling_scatter"
output_dir.mkdir(exist_ok=True, parents=True)

df_no_capitals = df[~df["nuts3_id"].isin(capital_nuts3)]

for y_col, years in years_by_variable.items():
    for year in years:
        fig, ax = plot_scaling_scatter(df, y_col=y_col, n_col="population", year=year)
        ax.set_title(f"{y_col} (teljes minta, {year})")
        fig.savefig(output_dir / f"{y_col}_full_{year}.png", dpi=150, bbox_inches="tight")
        plt.close(fig)

        fig, ax = plot_scaling_scatter(df_no_capitals, y_col=y_col, n_col="population", year=year)
        ax.set_title(f"{y_col} (fővárosok nélkül, {year})")
        fig.savefig(output_dir / f"{y_col}_no_capitals_{year}.png", dpi=150, bbox_inches="tight")
        plt.close(fig)

print(f"Mentve: {output_dir.relative_to(Path.cwd().parent)} ({len(list(output_dir.glob('*.png')))} fájl)")

In [ ]:
df_2023 = df[df["year"] == 2023]
numeric_cols = ["area", "population", "population_density", "gdp_total", "employment",
                "building_count", "building_footprint_sqm", "road_length_km",
                "pedestrian_network_length_km", "poi_count"]
df_2023[numeric_cols].corr()["area"].sort_values(ascending=False)

In [ ]:
df["road_length_per_sqrt_area"] = df["road_length_km"] / np.sqrt(df["area"])
df["pedestrian_per_sqrt_area"] = df["pedestrian_network_length_km"] / np.sqrt(df["area"])

In [ ]:
road_pop_fit = pd.DataFrame([fit_scaling_law(df[df["year"] == 2023], y_col="road_length_km", n_col="population")]).drop(columns="model")
road_area_fit = pd.DataFrame([fit_scaling_law(df[df["year"] == 2023], y_col="road_length_km", n_col="area")]).drop(columns="model")

road_comparison = pd.concat([road_pop_fit, road_area_fit], ignore_index=True)
road_comparison

In [ ]:
infra_y_cols = ["building_count", "building_footprint_sqm", "pedestrian_network_length_km"]

infra_comparison_rows = []
for y_col in infra_y_cols:
    pop_fit = fit_scaling_law(df[df["year"] == 2023], y_col=y_col, n_col="population")
    area_fit = fit_scaling_law(df[df["year"] == 2023], y_col=y_col, n_col="area")
    infra_comparison_rows.append(pop_fit)
    infra_comparison_rows.append(area_fit)

infra_comparison = pd.DataFrame(infra_comparison_rows).drop(columns="model")
infra_comparison

In [ ]:
poi_comparison = pd.DataFrame(
    [fit_scaling_law(df[df["year"] == 2023], y_col=c, n_col=n)
     for c in POI_COLS for n in ("population", "area")]
).drop(columns="model")
poi_comparison

In [ ]:
df["road_length_per_sqrt_area"] = df["road_length_km"] / np.sqrt(df["area"])

ln_norm_fit = fit_scaling_law(
    df[df["year"] == 2023],
    y_col="road_length_per_sqrt_area",
    n_col="population",
)
pd.DataFrame([ln_norm_fit]).drop(columns="model")

In [ ]:
df["pedestrian_per_sqrt_area"] = df["pedestrian_network_length_km"] / np.sqrt(df["area"])

pedestrian_norm_fit = fit_scaling_law(
    df[df["year"] == 2023],
    y_col="pedestrian_per_sqrt_area",
    n_col="population",
)
pd.DataFrame([pedestrian_norm_fit]).drop(columns="model")

In [ ]:
osm_fits = []

for y_col, n_col in [
    ("road_length_km", "population"),
    ("road_length_km", "area"),
    ("pedestrian_network_length_km", "population"),
    ("pedestrian_network_length_km", "area"),
    ("building_count", "population"),
    ("building_count", "area"),
    ("building_footprint_sqm", "population"),
    ("building_footprint_sqm", "area"),
*[(c, "population") for c in POI_COLS],
    *[(c, "area") for c in POI_COLS],
    ("road_length_per_sqrt_area", "population"),
    ("pedestrian_per_sqrt_area", "population"),
]:
    osm_fits.append(fit_scaling_law(df[df["year"] == 2023], y_col=y_col, n_col=n_col))

osm_comparison = pd.DataFrame(osm_fits).drop(columns="model")
osm_comparison

In [ ]:
osm_plot_pairs = [
    ("road_length_km", "population"),
    ("road_length_km", "area"),
    ("pedestrian_network_length_km", "population"),
    ("pedestrian_network_length_km", "area"),
    ("building_count", "population"),
    ("building_count", "area"),
    ("building_footprint_sqm", "population"),
    ("building_footprint_sqm", "area"),
*[(c, "population") for c in POI_COLS],
    *[(c, "area") for c in POI_COLS],
    ("road_length_per_sqrt_area", "population"),
    ("pedestrian_per_sqrt_area", "population"),
]

for y_col, n_col in osm_plot_pairs:
    fig, ax = plot_scaling_scatter(df, y_col=y_col, n_col=n_col, year=2023)
    ax.set_title(f"{y_col} ~ {n_col} (2023)")
    plt.show()

In [ ]:
df_no_capitals = df[~df["nuts3_id"].isin(capital_nuts3)]

osm_yn_pairs = [
    ("road_length_km", "population"),
    ("road_length_km", "area"),
    ("pedestrian_network_length_km", "population"),
    ("pedestrian_network_length_km", "area"),
    ("building_count", "population"),
    ("building_count", "area"),
    ("building_footprint_sqm", "population"),
    ("building_footprint_sqm", "area"),
*[(c, "population") for c in POI_COLS],
    *[(c, "area") for c in POI_COLS],
    ("road_length_per_sqrt_area", "population"),
    ("pedestrian_per_sqrt_area", "population"),
]

osm_fits_rows = []
for y_col, n_col in osm_yn_pairs:
    full_fit = fit_scaling_law(df[df["year"] == 2023], y_col=y_col, n_col=n_col)
    no_cap_fit = fit_scaling_law(df_no_capitals[df_no_capitals["year"] == 2023], y_col=y_col, n_col=n_col)

    row = {"y_variable": y_col, "n_variable": n_col}
    for key in ["beta", "beta_se", "beta_ci_low", "beta_ci_high", "r_squared", "n_regions", "breusch_pagan_pvalue"]:
        row[f"{key}_full"] = full_fit[key]
        row[f"{key}_no_capitals"] = no_cap_fit[key]
    osm_fits_rows.append(row)

osm_comparison = pd.DataFrame(osm_fits_rows)
osm_comparison

In [ ]:
output_dir = FIGURES_DIR / "scaling_scatter"
output_dir.mkdir(exist_ok=True, parents=True)

SHOW_COUNT = 2  # ennyi (y_col, n_col) párt jelenítünk meg ténylegesen

for i, (y_col, n_col) in enumerate(osm_yn_pairs):
    show = i < SHOW_COUNT

    fig, ax = plot_scaling_scatter(df, y_col=y_col, n_col=n_col, year=2023)
    ax.set_title(f"{y_col} ~ {n_col} (teljes minta, 2023)")
    fig.savefig(output_dir / f"{y_col}_vs_{n_col}_full_2023.png", dpi=150, bbox_inches="tight")
    if show:
        plt.show()
    else:
        plt.close(fig)

    fig, ax = plot_scaling_scatter(df_no_capitals, y_col=y_col, n_col=n_col, year=2023)
    ax.set_title(f"{y_col} ~ {n_col} (fővárosok nélkül, 2023)")
    fig.savefig(output_dir / f"{y_col}_vs_{n_col}_no_capitals_2023.png", dpi=150, bbox_inches="tight")
    if show:
        plt.show()
    else:
        plt.close(fig)

print(f"Mentve: {output_dir.relative_to(Path.cwd().parent)} ({len(list(output_dir.glob('*.png')))} fájl)")

In [ ]:
fig, ax = plot_scaling_scatter(df, y_col="area", n_col="population", year=2023)
ax.set_title("area ~ population (teljes minta, 2023)")
fig.savefig(FIGURES_DIR / "scaling_scatter" / "area_vs_population_full_2023.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
osm_output_path = RESULTS_DIR / "model_summaries" / "osm_scaling_comparison.csv"
osm_comparison.to_csv(osm_output_path, index=False)
print(f"Mentve: {osm_output_path.relative_to(Path.cwd().parent)}")

In [ ]:
import shutil

plots_dir = Path.cwd().parent / "plots"
plots_dir.mkdir(exist_ok=True, parents=True)

curated_figures = [
    "gdp_total_full_2023.png",
    "gdp_total_no_capitals_2023.png",
    "area_vs_population_full_2023.png",
    "road_length_km_vs_area_full_2023.png",
    "road_length_per_sqrt_area_vs_population_full_2023.png",
    "pedestrian_per_sqrt_area_vs_population_full_2023.png",
    "building_count_vs_population_full_2023.png",
    "building_count_vs_area_full_2023.png",
]
# Add the innovation-POI figures (names derived from POI_COLS)
curated_figures += [f"{c}_vs_population_full_2023.png" for c in POI_COLS]

source_dir = FIGURES_DIR / "scaling_scatter"

for filename in curated_figures:
    src = source_dir / filename
    if src.exists():
        shutil.copy(src, plots_dir / filename)
    else:
        print(f"[hiányzik] {filename}")

print(f"Másolva: {plots_dir.relative_to(Path.cwd().parent)} ({len(list(plots_dir.glob('*.png')))} fájl)")